# Generate Head Boundary Conditions for 2D transect ATS - Naches

Head extracted from Zhi's 3D ATS simulation for Naches

- file `global/cfs/cdirs/m1800/naches_run2_share/Naches-2`
- Information
    - "Time": from 11224 to 16059; unit is day;
        - 11400 = 85+365x31 --> 2011.03.26
        - 12631 = 221+365x34 --> 2014.08.09

- file `global/cfs/cdirs/m1800/naches_run2_share/Naches-3`
- Information
    - "Time": from 11224 to 16059; unit is day;
        - 12600 = 190+365x34 --> 2014.07.09
        - 15969 = 274+365x43 --> 2023.10.01

Output of this script

- constant head at the starting and end points -> to drive the run0
- head at a typical year at the starting and end points -> to drive the run1
- transient head

**File History**

update 2026/1/29
- previously, it was a two-step extraction
    - 1. extract point water head on nersc
    - 2. process cyclic spinup and transient
- now, merge two steps into this notebook
    - merge with `10-Projects/2025-RCSFA-HillslopeFire/MaterialsData/OakCreek_from_sundar/ats_WTD_pressure_based_Aug1_2024.ipynb` on NERSC

update 2025/10/13
- update `config.json`. Mainly revise the model run pipeline.

update 2025/8/32
- add `config.json`

update 2025/8/7
- correct site name to NF01
- a cleaned version putting all input data and notebooks together

In [ ]:
%load_ext autoreload
%autoreload 2

# Parameters and data sources

In [ ]:
# Parameters cell -- schema-v2 date-based configuration
from config_utils import load_config, phase_dates, phase_label, phase_names, phase_period, noleap_day_of_year, phase_forcing_dir, full_timeline_forcing_dir

config = load_config('config.json')
case = config['case']
watershed_name = case['watershed_name']
hucs = case['hucs']
site_name = case['site_name']
meshsize_nx = case['meshsize_nx']

spinup_dates = phase_dates(config, 'spinup')
prefire_dates = phase_dates(config, 'prefire_transient')
postfire_dates = phase_dates(config, 'postfire_transient') if 'postfire_transient' in config else []
spinup_label = phase_label(config, 'spinup')
prefire_label = phase_label(config, 'prefire_transient')
postfire_label = phase_label(config, 'postfire_transient') if postfire_dates else None
forcing_spinup_dir = phase_forcing_dir(config, 'spinup')
forcing_prefire_dir = phase_forcing_dir(config, 'prefire_transient')
forcing_postfire_dir = phase_forcing_dir(config, 'postfire_transient') if postfire_dates else None
forcing_full_dir = full_timeline_forcing_dir(config)
for _d in (forcing_spinup_dir, forcing_prefire_dir, forcing_postfire_dir, forcing_full_dir):
    if _d is not None: _d.mkdir(parents=True, exist_ok=True)

# Temporary aliases keep unchanged downstream cells executable while their
# date selections use the phase-specific date lists above.
start_year_spinup = spinup_dates[0].year
end_year_spinup = spinup_dates[-1].year
nyears_steadystate_spinup = config['spinup']['steady_state_years']
nyears_cyclic_spinup = config['spinup']['cyclic_years']
start_year_transient = prefire_dates[0].year
end_year_transient = prefire_dates[-1].year
run = config['prefire_transient']['elm_run']


In [ ]:
outputs={}

## extract point raw data from 3D ATS simulation

In [ ]:
import os, sys
import numpy as np
import matplotlib.pyplot as plt
import h5py as h5

import scipy.signal
from datetime import datetime, timedelta
import pandas as pd

import ats_xdmf as xdmf
import time
import random
import pandas
import os

from scipy.io import loadmat
import shapely
from shapely.geometry import Point, LineString, Polygon, box, mapping
import geopandas as gpd

### Config 3D ATS-flow results

In [ ]:
# Define the output model directory, whre ats_vis data (.h5) are located
# model_dir = '/global/cfs/cdirs/m1800/naches_run2_share/Naches-3'

# Define the Parameters AND verify with the XML
rho = 997 # density of water, kg m^-3
g = 9.80665 # gravity, m s^-2
patm = 101325 # atmopsheric pressure, Pascals

# Define raw output, and skip raw point data extraction if detect it
outputs['tmp_BChead_raw'] = [
    f'../data-processed/{site_name}/tmp_bc_startend_raw.naches-2.h5',
    f'../data-processed/{site_name}/tmp_bc_startend_raw.naches-3.h5'
]

## Process start/end point water head raw data
- for hillslope spinup and transient simulations

In [ ]:
# Read datasets from all HDF5 files in the list
raw_data = []  # Store data from each file temporarily

for i, fname_headbc_raw in enumerate(outputs['tmp_BChead_raw']):
    print(f"\nReading file {i+1}/{len(outputs['tmp_BChead_raw'])}: {fname_headbc_raw}")
    
    with h5.File(fname_headbc_raw, "r") as hdf:
        file_data = {
            'time': hdf["Time"][:],
            'startpt_head_subsrf_vis': hdf["startpt_head_subsrf_vis"][:],
            'endpt_head_subsrf_vis': hdf["endpt_head_subsrf_vis"][:],
            'startpt_head_srf_vis': hdf["startpt_head_srf_vis"][:],
            'endpt_head_srf_vis': hdf["endpt_head_srf_vis"][:]
        }
        raw_data.append(file_data)
        
        # Print shapes to confirm
        print(f"  Time shape: {file_data['time'].shape}")
        print(f"  Time range: {file_data['time'][0]} to {file_data['time'][-1]} days")
        print(f"  Start point subsurface head shape: {file_data['startpt_head_subsrf_vis'].shape}")
        print(f"  End point subsurface head shape: {file_data['endpt_head_subsrf_vis'].shape}")
        print(f"  Start point surface head shape: {file_data['startpt_head_srf_vis'].shape}")
        print(f"  End point surface head shape: {file_data['endpt_head_srf_vis'].shape}")

print(f"\nTotal files read: {len(raw_data)}")

In [ ]:
# Process overlap and merge time/startpt_head/endpt_head from multiple files
# Assuming files are ordered chronologically and may have overlapping time ranges

# For now, simple concatenation (you can refine the overlap handling logic)
time_list = []
startpt_head_subsrf_list = []
endpt_head_subsrf_list = []
startpt_head_srf_list = []
endpt_head_srf_list = []

for i, file_data in enumerate(raw_data):
    if i == 0:
        # Add all data from first file
        time_list.append(file_data['time'])
        startpt_head_subsrf_list.append(file_data['startpt_head_subsrf_vis'])
        endpt_head_subsrf_list.append(file_data['endpt_head_subsrf_vis'])
        startpt_head_srf_list.append(file_data['startpt_head_srf_vis'])
        endpt_head_srf_list.append(file_data['endpt_head_srf_vis'])
    else:
        # For subsequent files, find where to start to avoid overlap
        # Assuming time is monotonically increasing
        last_time = time_list[-1][-1]
        mask = file_data['time'] > last_time
        
        if np.any(mask):
            time_list.append(file_data['time'][mask])
            startpt_head_subsrf_list.append(file_data['startpt_head_subsrf_vis'][mask])
            endpt_head_subsrf_list.append(file_data['endpt_head_subsrf_vis'][mask])
            startpt_head_srf_list.append(file_data['startpt_head_srf_vis'][mask])
            endpt_head_srf_list.append(file_data['endpt_head_srf_vis'][mask])
            print(f"File {i+1}: Added {np.sum(mask)} points after time {last_time}")
        else:
            print(f"File {i+1}: No new data points (all overlap with previous data)")

# Concatenate all arrays
time = np.concatenate(time_list)
startpt_head_subsrf = np.concatenate(startpt_head_subsrf_list)
endpt_head_subsrf = np.concatenate(endpt_head_subsrf_list)
startpt_head_srf = np.concatenate(startpt_head_srf_list)
endpt_head_srf = np.concatenate(endpt_head_srf_list)

# Print merged shapes to confirm
print(f"\nMerged data:")
print(f"Time shape: {time.shape}")
print(f"Start point subsurface head shape: {startpt_head_subsrf.shape}")
print(f"End point subsurface head shape: {endpt_head_subsrf.shape}")
print(f"Start point surface head shape: {startpt_head_srf.shape}")
print(f"End point surface head shape: {endpt_head_srf.shape}")
print(f"Time range: {time[0]} to {time[-1]} days")

In [ ]:
# Create a plot with two lines
plt.figure(figsize=(10, 5))

plt.plot(time, startpt_head_subsrf, label="Start Point Head", linestyle="-", color="blue")
plt.plot(time, endpt_head_subsrf, label="End Point Head", linestyle="--", color="red")

# Formatting the plot
plt.xlabel("Time [days]")
plt.ylabel("Head Value")
plt.title("Start and End Point Head over Time - subsurface vis")
plt.legend()
plt.grid(True)

# Show the plot
plt.show()

# Create a plot with two lines
plt.figure(figsize=(10, 5))

plt.plot(time, startpt_head_srf, label="Start Point Head", linestyle="-", color="blue")
plt.plot(time, endpt_head_srf, label="End Point Head", linestyle="--", color="red")

# Formatting the plot
plt.xlabel("Time [days]")
plt.ylabel("Head Value")
plt.title("Start and End Point Head over Time - surface vis")
plt.legend()
plt.grid(True)

# Show the plot
plt.show()

## Merge subsurface vis based and surface vis based water table depth and poneded water depth

accomodate non-hydrostatic boundary conditions
- for subsurface BC head
    - simply use startpt_head_subsrf and endpt_head_subsrf, so no correct needed
- for surface BC head
    - based on startpt_head_srf and endpt_head_srf,
    - if xxx_head_srf is positive
        - keep the value
    - if xxx_head_srf is ZERO
        - replace with values from xxx_head_subsrf.

In [ ]:
# Process subsurface and surface BC head data
print("\nProcessing BC head data...")

# Optionally adapt the outlet series when an earlier v4 cut wrote metadata.
# Without that file or its delta_z_BC_m field, preserve the legacy behavior.
cut_summary_path = f'../data-processed/{site_name}/cut_transect_{site_name}_summary.csv'
if os.path.exists(cut_summary_path):
    cut_summary = pd.read_csv(cut_summary_path)
    if len(cut_summary) == 1 and 'delta_z_BC_m' in cut_summary.columns and pd.notna(cut_summary.loc[0, 'delta_z_BC_m']):
        delta_z_BC = float(cut_summary.loc[0, 'delta_z_BC_m'])
        endpt_head_subsrf = endpt_head_subsrf - delta_z_BC
        endpt_head_srf = endpt_head_srf - delta_z_BC
        if min(endpt_head_subsrf.min(), endpt_head_srf.min()) < -1.e-6:
            import warnings
            warnings.warn('Cut outlet correction produced negative ponded depths; retaining them as groundwater table depths for the water-head boundary condition.', UserWarning)
        print(f'Applied optional outlet delta_z_BC={delta_z_BC:.3f} m from {cut_summary_path}.')
    else:
        print(f'No usable delta_z_BC_m in {cut_summary_path}; no outlet correction applied.')
else:
    print('No cut-transect metadata found; no outlet correction applied.')

startpt_head = startpt_head_subsrf.copy()
endpt_head = endpt_head_subsrf.copy()

# For surface BC head - replace zeros with subsurface values
startpt_head_srf_corrected = startpt_head_srf.copy()
endpt_head_srf_corrected = endpt_head_srf.copy()

# Replace zero values in surface head with subsurface values
mask_start_zero = startpt_head_srf == 0
mask_end_zero = endpt_head_srf == 0

startpt_head_srf_corrected[mask_start_zero] = startpt_head_subsrf[mask_start_zero]
endpt_head_srf_corrected[mask_end_zero] = endpt_head_subsrf[mask_end_zero]

# Print statistics
print(f"Start point surface head: {np.sum(mask_start_zero)} zero values replaced with subsurface values")
print(f"End point surface head: {np.sum(mask_end_zero)} zero values replaced with subsurface values")

# Plot comparison
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# Subsurface head
axes[0, 0].plot(time, startpt_head_subsrf, label="Start Point", color="blue")
axes[0, 0].plot(time, endpt_head_subsrf, label="End Point", color="red")
axes[0, 0].set_xlabel("Time [days]")
axes[0, 0].set_ylabel("Head [m]")
axes[0, 0].set_title("Subsurface Head (no correction)")
axes[0, 0].legend()
axes[0, 0].grid(True)

# Surface head - original
axes[0, 1].plot(time, startpt_head_srf, label="Start Point", color="blue")
axes[0, 1].plot(time, endpt_head_srf, label="End Point", color="red")
axes[0, 1].set_xlabel("Time [days]")
axes[0, 1].set_ylabel("Head [m]")
axes[0, 1].set_title("Surface Head (original)")
axes[0, 1].legend()
axes[0, 1].grid(True)

# Surface head - corrected
axes[1, 0].plot(time, startpt_head_srf_corrected, label="Start Point", color="blue")
axes[1, 0].plot(time, endpt_head_srf_corrected, label="End Point", color="red")
axes[1, 0].set_xlabel("Time [days]")
axes[1, 0].set_ylabel("Head [m]")
axes[1, 0].set_title("Surface Head (corrected - zeros replaced)")
axes[1, 0].legend()
axes[1, 0].grid(True)

# Difference plot
axes[1, 1].plot(time, startpt_head_srf_corrected - startpt_head_srf, label="Start Point Diff", color="blue", alpha=0.7)
axes[1, 1].plot(time, endpt_head_srf_corrected - endpt_head_srf, label="End Point Diff", color="red", alpha=0.7)
axes[1, 1].set_xlabel("Time [days]")
axes[1, 1].set_ylabel("Head Difference [m]")
axes[1, 1].set_title("Correction Applied (corrected - original)")
axes[1, 1].legend()
axes[1, 1].grid(True)

plt.tight_layout()
plt.show()

## Split raw for spinup and transient

In [ ]:
# Select exact inclusive no-leap phase dates from the 3-D ATS time axis.
# The raw axis uses day 10950 for 2010-01-01 and contains no leap days.
def _raw_time_for_date(day):
    return 10950 + 365 * (day.year - 2010) + noleap_day_of_year(day)

def _select_phase_head(days):
    requested_time = np.array([_raw_time_for_date(day) for day in days])
    indices = np.searchsorted(time, requested_time)
    if (indices >= len(time)).any() or not np.array_equal(time[indices], requested_time):
        raise ValueError('3-D ATS head record does not cover requested phase dates')
    return (time[indices], startpt_head[indices], endpt_head[indices],
            startpt_head_srf_corrected[indices], endpt_head_srf_corrected[indices])

(time_spinup, startpt_head_spinup, endpt_head_spinup,
 startpt_head_srf_corrected_spinup, endpt_head_srf_corrected_spinup) = _select_phase_head(spinup_dates)
(time_transient, startpt_head_transient, endpt_head_transient,
 startpt_head_srf_corrected_transient, endpt_head_srf_corrected_transient) = _select_phase_head(prefire_dates)

startdate_spinup, enddate_spinup = spinup_label.split('_')
startdate_transient, enddate_transient = prefire_label.split('_')
datetime_spinup = pd.DatetimeIndex([pd.Timestamp(day) for day in spinup_dates])
datetime_transient = pd.DatetimeIndex([pd.Timestamp(day) for day in prefire_dates])
print('spinup:', startdate_spinup, enddate_spinup, len(datetime_spinup))
print('prefire:', startdate_transient, enddate_transient, len(datetime_transient))


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 6))
ax = axes.flatten()

# Subsurface BCs - spinup
ax[0].plot(datetime_spinup, startpt_head_spinup, label="Left Boundary Head", linestyle="-", color="blue")
ax[0].plot(datetime_spinup, endpt_head_spinup, label="Right Boundary Head", linestyle="--", color="red")
ax[0].set_xlabel("Time")
ax[0].set_ylabel("Head Value")
ax[0].set_title("Flow subsurface BCs - spinup raw")
ax[0].legend()

# Subsurface BCs - transient
ax[1].plot(datetime_transient, startpt_head_transient, label="Left Boundary Head", linestyle="-", color="blue")
ax[1].plot(datetime_transient, endpt_head_transient, label="Right Boundary Head", linestyle="--", color="red")
ax[1].set_xlabel("Time")
ax[1].set_ylabel("Head Value")
ax[1].set_title("Flow subsurface BCs - transient raw")
ax[1].legend()

# Surface BCs - spinup
ax[2].plot(datetime_spinup, startpt_head_srf_corrected_spinup, label="Left Boundary Head", linestyle="-", color="blue")
ax[2].plot(datetime_spinup, endpt_head_srf_corrected_spinup, label="Right Boundary Head", linestyle="--", color="red")
ax[2].set_xlabel("Time")
ax[2].set_ylabel("Head Value")
ax[2].set_title("Flow surface BCs - spinup raw")
ax[2].legend()

# Surface BCs - transient
ax[3].plot(datetime_transient, startpt_head_srf_corrected_transient, label="Left Boundary Head", linestyle="-", color="blue")
ax[3].plot(datetime_transient, endpt_head_srf_corrected_transient, label="Right Boundary Head", linestyle="--", color="red")
ax[3].set_xlabel("Time")
ax[3].set_ylabel("Head Value")
ax[3].set_title("Flow surface BCs - transient raw")
ax[3].legend()

plt.tight_layout()
plt.show()

## calculate typical year head BC for spinup

In [ ]:
# The configured source period is a complete Oct--Sep water-year sequence.
# It is already no-leap and must contain an integer number of 365-day years.
days_in_year = 365
complete_years, remaining_days = divmod(len(spinup_dates), days_in_year)
if remaining_days:
    raise ValueError(f'Spinup source period has {len(spinup_dates)} days; expected complete no-leap water years.')
if complete_years == 0:
    raise ValueError('Spinup source period must contain at least one complete water year.')
if startpt_head_spinup.shape[0] != len(spinup_dates):
    raise ValueError('Selected spinup boundary-head records do not match the configured spinup dates.')

startpt_head_reshaped = startpt_head_spinup.reshape(complete_years, days_in_year)
endpt_head_reshaped = endpt_head_spinup.reshape(complete_years, days_in_year)
startpt_head_srf_corrected_reshaped = startpt_head_srf_corrected_spinup.reshape(complete_years, days_in_year)
endpt_head_srf_corrected_reshaped = endpt_head_srf_corrected_spinup.reshape(complete_years, days_in_year)
print(f'Using {complete_years} complete no-leap water years ({len(spinup_dates)} days) for typical-year calculation.')

# Compute the mean across years for each day of a typical year
# Subsurface head
startpt_head_typical = np.mean(startpt_head_reshaped, axis=0)
endpt_head_typical = np.mean(endpt_head_reshaped, axis=0)

# Surface head (corrected)
startpt_head_srf_corrected_typical = np.mean(startpt_head_srf_corrected_reshaped, axis=0)
endpt_head_srf_corrected_typical = np.mean(endpt_head_srf_corrected_reshaped, axis=0)

In [ ]:
# Create an array for days in a typical year
typical_days = np.arange(365)

# Create subplots
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Subplot 1: Subsurface head
axes[0].plot(typical_days, startpt_head_typical, label="Start Point Head", color="blue")
axes[0].plot(typical_days, endpt_head_typical, label="End Point Head", color="red", linestyle="--")
axes[0].set_xlabel("Day of the Year")
axes[0].set_ylabel("Head Value")
axes[0].set_title("Average Subsurface Head for a Typical Year")
axes[0].legend()

# Subplot 2: Surface head (corrected)
axes[1].plot(typical_days, startpt_head_srf_corrected_typical, label="Start Point Head", color="blue")
axes[1].plot(typical_days, endpt_head_srf_corrected_typical, label="End Point Head", color="red", linestyle="--")
axes[1].set_xlabel("Day of the Year")
axes[1].set_ylabel("Head Value")
axes[1].set_title("Average Surface Head for a Typical Year (Corrected)")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Smoothing function from watershed-workflow [to-do, direct import from watershed-workflow?]
def smooth_array(data, method, axis=0, **kwargs):
    """Smooths fixed-interval time-series data using a Sav-Gol filter from scipy."""
    if method is True:
        method = 'savgol_filter'

    if method == 'savgol_filter':
        if 'window_length' not in kwargs:
            kwargs['window_length'] = 61  # Default window length for Savitzky-Golay filter
        if 'polyorder' not in kwargs:
            kwargs['polyorder'] = 2  # Default polynomial order for Savitzky-Golay filter
        if 'mode' not in kwargs:
            kwargs['mode'] = 'wrap'  # Wrap around for cyclic behavior
        return scipy.signal.savgol_filter(data, axis=axis, **kwargs)
    elif method == 'convolve':
        if 'window' not in kwargs:
            kwargs['window'] = 'hann'
        if 'Nx' not in kwargs:
            kwargs['Nx'] = 50
        win = scipy.signal.windows.get_window(**kwargs)
        win = win / win.sum()
        assert (len(data.shape) == 3 and axis == 0)
        data_new = np.empty_like(data)
        for i in range(data.shape[1]):
            for j in range(data.shape[2]):
                data_new[:, i, j] = scipy.signal.convolve(data[:, i, j],
                                                          win)[len(win) // 2:-len(win) // 2 + 1]
        return data_new
    else:
        raise ValueError(f'Invalid smooth method {method}')

In [ ]:
# Define smoothing parameters (from watershed-workflow)
smooth_kwargs = dict(window_length=181, polyorder=2)

# Repeat for nyears_cyclic_spinup - subsurface head
startpt_head_repeated = np.tile(startpt_head_typical, nyears_cyclic_spinup)
endpt_head_repeated = np.tile(endpt_head_typical, nyears_cyclic_spinup)

# Repeat for nyears_cyclic_spinup - surface head (corrected)
startpt_head_srf_corrected_repeated = np.tile(startpt_head_srf_corrected_typical, nyears_cyclic_spinup)
endpt_head_srf_corrected_repeated = np.tile(endpt_head_srf_corrected_typical, nyears_cyclic_spinup)

typical_days_repeated = np.arange(365 * nyears_cyclic_spinup)

# Apply the smoothing - subsurface head
startpt_head_smooth_repeated = smooth_array(startpt_head_repeated, method='savgol_filter', window_length=181, polyorder=2, mode='wrap')
endpt_head_smooth_repeated = smooth_array(endpt_head_repeated, method='savgol_filter', window_length=181, polyorder=2, mode='wrap')

# Apply the smoothing - surface head (corrected)
startpt_head_srf_corrected_smooth_repeated = smooth_array(startpt_head_srf_corrected_repeated, method='savgol_filter', window_length=181, polyorder=2, mode='wrap')
endpt_head_srf_corrected_smooth_repeated = smooth_array(endpt_head_srf_corrected_repeated, method='savgol_filter', window_length=181, polyorder=2, mode='wrap')

# Create subplots
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Subplot 1: Subsurface head
axes[0].plot(typical_days_repeated, startpt_head_smooth_repeated, label="Start Point Head (Smoothed)", color="blue")
axes[0].plot(typical_days_repeated, endpt_head_smooth_repeated, label="End Point Head (Smoothed)", color="red", linestyle="--")
axes[0].set_xlabel("Day of the Year")
axes[0].set_ylabel("Head Value")
axes[0].set_title("Average Subsurface Head for a Typical Year (Smoothed)")
axes[0].legend()
axes[0].grid(True)

# Subplot 2: Surface head (corrected)
axes[1].plot(typical_days_repeated, startpt_head_srf_corrected_smooth_repeated, label="Start Point Head (Smoothed)", color="blue")
axes[1].plot(typical_days_repeated, endpt_head_srf_corrected_smooth_repeated, label="End Point Head (Smoothed)", color="red", linestyle="--")
axes[1].set_xlabel("Day of the Year")
axes[1].set_ylabel("Head Value")
axes[1].set_title("Average Surface Head for a Typical Year (Smoothed, Corrected)")
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

## generate head BC for run0, run1, and run2

In [ ]:
# for run0, use mean head to drive steady-state spinup
startpt_head_smooth_repeated_mean = np.mean(startpt_head_smooth_repeated)
endpt_head_smooth_repeated_mean = np.mean(endpt_head_smooth_repeated)

print(startpt_head_smooth_repeated_mean)
print(endpt_head_smooth_repeated_mean)

In [ ]:
time_run1 = np.arange(365 * nyears_cyclic_spinup) * 86400
for path, subsurface, surface, prefix in [(forcing_spinup_dir / 'startpt_head.h5', startpt_head_smooth_repeated, startpt_head_srf_corrected_smooth_repeated, 'startpt'), (forcing_spinup_dir / 'endpt_head.h5', endpt_head_smooth_repeated, endpt_head_srf_corrected_smooth_repeated, 'endpt')]:
    with h5.File(path, 'w') as hdf:
        hdf.create_dataset('Time', data=time_run1); hdf.create_dataset(f'{prefix}_head_subsrf', data=subsurface); hdf.create_dataset(f'{prefix}_head_srf', data=surface)
outputs['BChead_start_spinup_filename_site'] = str(forcing_spinup_dir / 'startpt_head.h5'); outputs['BChead_end_spinup_filename_site'] = str(forcing_spinup_dir / 'endpt_head.h5')


In [ ]:
time_run2 = (len(time_run1) + np.arange(len(prefire_dates))) * 86400
for path, subsurface, surface, prefix in [(forcing_prefire_dir / 'startpt_head.h5', startpt_head_transient, startpt_head_srf_corrected_transient, 'startpt'), (forcing_prefire_dir / 'endpt_head.h5', endpt_head_transient, endpt_head_srf_corrected_transient, 'endpt')]:
    with h5.File(path, 'w') as hdf:
        hdf.create_dataset('Time', data=time_run2); hdf.create_dataset(f'{prefix}_head_subsrf', data=subsurface); hdf.create_dataset(f'{prefix}_head_srf', data=surface)


In [ ]:
# merged data for ats-pflotran transient, which restarted from ats-pflotran spinup run

In [ ]:
postfire_time = np.array([], dtype=int); post = None
if postfire_dates:
    _, ps, pe, pss, pes = _select_phase_head(postfire_dates); post = (ps, pe, pss, pes); postfire_time = (len(time_run1) + len(time_run2) + np.arange(len(postfire_dates))) * 86400
    for path, subsurface, surface, prefix in [(forcing_postfire_dir / 'startpt_head.h5', ps, pss, 'startpt'), (forcing_postfire_dir / 'endpt_head.h5', pe, pes, 'endpt')]:
        with h5.File(path, 'w') as hdf:
            hdf.create_dataset('Time', data=postfire_time); hdf.create_dataset(f'{prefix}_head_subsrf', data=subsurface); hdf.create_dataset(f'{prefix}_head_srf', data=surface)
times = np.concatenate((time_run1, time_run2, postfire_time))
start_sub = np.concatenate((startpt_head_smooth_repeated, startpt_head_transient) + ((post[0],) if post else ()))
end_sub = np.concatenate((endpt_head_smooth_repeated, endpt_head_transient) + ((post[1],) if post else ()))
start_srf = np.concatenate((startpt_head_srf_corrected_smooth_repeated, startpt_head_srf_corrected_transient) + ((post[2],) if post else ()))
end_srf = np.concatenate((endpt_head_srf_corrected_smooth_repeated, endpt_head_srf_corrected_transient) + ((post[3],) if post else ()))
if not np.all(np.diff(times) == 86400) or any(len(x) != len(times) for x in (start_sub, end_sub, start_srf, end_srf)): raise ValueError('BC-head full timeline is inconsistent')
for path, subsurface, surface, prefix in [(forcing_full_dir / 'startpt_head.h5', start_sub, start_srf, 'startpt'), (forcing_full_dir / 'endpt_head.h5', end_sub, end_srf, 'endpt')]:
    with h5.File(path, 'w') as hdf:
        hdf.create_dataset('Time', data=times); hdf.create_dataset(f'{prefix}_head_subsrf', data=subsurface); hdf.create_dataset(f'{prefix}_head_srf', data=surface)
outputs['BChead_start_full_timeline_filename_site'] = str(forcing_full_dir / 'startpt_head.h5'); outputs['BChead_end_full_timeline_filename_site'] = str(forcing_full_dir / 'endpt_head.h5')
print('\nBC-head forcing summary'); print(f'  spinup source period: {spinup_label} ({len(spinup_dates)} days; {complete_years} water years)'); print(f'  cyclic ATS spinup: {nyears_cyclic_spinup} years ({len(time_run1)} days)'); print(f'  prefire transient: {prefire_label} ({len(prefire_dates)} days)'); print(f'  postfire transient: {postfire_label} ({len(postfire_dates)} days)' if postfire_dates else '  postfire transient: not configured'); print(f'  full timeline: {len(times)} days; time = {times[0]} to {times[-1]} s')


In [ ]:
outputs

In [ ]:
# Postfire phase files and canonical full-timeline BC-head files are written in cell 32.
